# Long Story Short: YouTube Audit Tool — Demo

This notebook walks through a complete audit experiment using the `longstoryshort` package.

**What it does:**
1. Configures and launches a headless Chrome browser
2. Watches a set of *seed videos* to build a YouTube user profile
3. Follows the autoplay chain for a given number of hops
4. At each hop, collects sidebar recommendations (long-form) or preloaded videos (short-form)
5. Returns a structured report of everything collected

> **Note:** Pre-saved sample outputs are shown inline below each step. You can re-run the cells with your own seed video IDs to collect live data.

## Setup

Install dependencies from the repo root under your environement:

```bash
# if using virtual env: source <virtual_environement>/bin/activate
pip install -e .          # installs the longstoryshort package
# or: pip install -r requirements.txt
```

No separate ChromeDriver download is needed — Selenium 4.6+ manages the driver automatically.

In [ ]:
import json
from longstoryshort import YouTubeAuditor

In [ ]:
%pip show longstoryshort

Name: longstoryshort
Version: 0.1.0
Summary: Automated YouTube recommendation audit tool for studying long- vs. short-form video recommendations
Home-page: 
Author: 
Author-email: 
License-Expression: Apache-2.0
Location: /Users/liangzhenning/Desktop/longstoryshort-yt-audit/.venv/lib/python3.11/site-packages
Editable project location: /Users/liangzhenning/Desktop/longstoryshort-yt-audit
Requires: selenium
Required-by: 
Note: you may need to restart the kernel to use updated packages.


## 1. Configure the experiment

Set your seed video IDs and experiment parameters here.

- **Seed videos** are watched first to build the recommendation profile.
- **Hops** control how far down the autoplay chain we follow.
- **Watch time** is the number of seconds spent on each video.

In [ ]:
# --- Experiment parameters ---
SEED_VIDEO_IDS = [
    "dQw4w9WgXcQ",   # seed 1
    "9bZkp7q19f0",   # seed 2
    "kJQP7kiw5Fk",   # seed 3
]

MODE         = "long"   # "long" for regular videos, "short" for YouTube Shorts
WATCH_TIME   = 10       # seconds to watch each video
HOPS         = 10       # number of autoplay steps to follow

## 2. Launch the browser

The auditor follows a two-phase setup:
- `configure_browser()` — set browser options (headless, incognito, ad blocker)
- `launch_browser()` — start the WebDriver and prepare the session

Set `headless=False` to watch the browser navigate in real time.
Set `incognito=True` to launch the browser in incognito mode.

In [ ]:
auditor = YouTubeAuditor(log_file_path="demo_audit.log")

auditor.configure_browser(browser_type="Chrome", headless=False, incognito=False)

print("configure ready")

Browser ready.


### (Optional, Suggested) Configure extensions and browser version
Since May 2025, the Chrome team stopped supporting the `--load-extension` switch in newer release.
Hence, it is unable to load .crx extensions when using the latest chrome driver.

Alternatively, you may use [Chrome for Testing](https://googlechromelabs.github.io/chrome-for-testing/), or "cft" in short, to load extensions or access older Chrome versions.
For more detail, please check the [article](https://developer.chrome.com/blog/chrome-for-testing).

In [ ]:
import os

# forcing selenium to download chrome for testing
os.environ["SE_FORCE_BROWSER_DOWNLOAD"] = "true"
os.environ["SE_CACHE_PATH"] = "~/.cache"
version = "stable" # or any other newer or older versions


extension_path = True | "path_to_extension"
# set to True to load all .crx in the current working directory
# set to a folder to load all .crx files in the folder
# or, set to path pointing to a single .crx file
try:
    auditor.configure_browser(
        browser_type="Chrome",
        browser_version=version,
        headless=False,
        incognito=False,
        extension=extension_path,
    )
except FileNotFoundError as e:
    print(f"Extension not found: {e}")


# # Or, if you have downloaded a driver
# cft_path = "path_to_app"
# auditor.configure_browser(
#     browser_type="Chrome", binary_location=cft_path, extension=extension_path
# )

['...']

### (Optional) Use extensions in incognito mode 
Extensions are disabled by default in incognito mode, but can be toggled on in Chrome profile (check [doc](https://support.google.com/chrome/a/answer/13130396?hl=en)).

You may use that specific profile for your project by linking the webdriver to the profile directory.


In [ ]:
# # Set up your Profile and enable the desired extensions beforehand.

# # Look for the Profile's parent path. It usually looks like this:
# ## Windows: C:\Users\<YourUsername>\AppData\Local\Google\Chrome\User Data\
# ## macOS: /Users/<YourUsername>/Library/Application Support/Google/Chrome/
# data_dir = "path_to_profile"
# profile_name = "name_of_profile" # e.g. Default, Profile 1

# arguments = [f"--user-data-dir={data_dir}", f"--profile-directory={profile_name}"]

# auditor.configure_browser(
#     browser_type="Chrome", incognito=True, custom_argument=arguments
# ) # no need to load extensions in this case

### Launch the browser after completing all configuration

In [ ]:
failed = auditor.launch_browser(mode=MODE, max_duration=WATCH_TIME)
if failed:
    raise RuntimeError("Browser failed to launch — check demo_audit.log")

print("Browser ready.")

## 3. (Optional) Log in

Logging in lets you audit personalized recommendations for a specific Google account.
Skip this cell to run an anonymous (logged-out) audit.

> Use a dedicated research account — not your personal Google account.

In [ ]:
# success = auditor.log_in(username="research@gmail.com", password="...")
# if not success:
#     print("Login failed — running in anonymous mode")

## 4. Train on seed videos

`train()` watches each seed video for `WATCH_TIME` seconds.
This simulates a user who has been consuming a particular type of content,
nudging YouTube's algorithm to surface related recommendations.

In [ ]:
print(f"Training on {len(SEED_VIDEO_IDS)} seed videos ({WATCH_TIME}s each)...")
failed = auditor.train(SEED_VIDEO_IDS)
if failed:
    auditor.clean_up()
    raise RuntimeError("Training failed — check demo_audit.log")

print("Training complete.")

Training on 3 seed videos (10s each)...
Training complete.


## 5. Collect recommendations

`collect_play_next()` presses the "play next" control (`Shift+N` for long-form,
`↓` for Shorts) and records:
- **autoplay path** — the sequence of URLs visited
- **sidebar recommendations** — related videos listed alongside each long-form video
- **preloaded recommendations** — videos buffered for swipe in Shorts

In [ ]:
print(f"Following autoplay chain for {HOPS} hops...")
failed = auditor.collect_play_next(collect_video_num=HOPS)
if failed:
    print("Collection ended early — check demo_audit.log for details.")
else:
    print("Collection complete.")

Following autoplay chain for 10 hops...
Collection complete.


## 6. Inspect the results

`report()` returns a dictionary with all collected data.
The sample output below was collected from a real anonymous run in long-form mode.

In [ ]:
results = auditor.report()
print(json.dumps(results, indent=2))

{
  "training_ids": [
    "dQw4w9WgXcQ",
    "9bZkp7q19f0"
  ],
  "seed_id": "kJQP7kiw5Fk",
  "player_mode": "long",
  "max_duration": 10,
  "recommendations": {
    "autoplay_rec": [
      "https://www.youtube.com/watch?v=Jma4nCMpaQM",
      "https://www.youtube.com/watch?v=iNJG3xbw-2E",
      "https://www.youtube.com/watch?v=vj5FtwDgmSs",
      "https://www.youtube.com/watch?v=pU7culxnZT0",
      "https://www.youtube.com/watch?v=ZWuXzQ4D9u8",
      "https://www.youtube.com/watch?v=C-fexNlzMtQ",
      "https://www.youtube.com/watch?v=zdjo712qnyE",
      "https://www.youtube.com/watch?v=9aBG9yGcWSc",
      "https://www.youtube.com/watch?v=ga95aB7CAPs",
      "https://www.youtube.com/watch?v=dZuVOViXOq0"
    ],
    "sidebar_rec": [
      [],
      [],
      [],
      [],
      [],
      [],
      [],
      [],
      [],
      []
    ],
    "preload_rec": [],
    "restricted": []
  }
}


## 7. Save results

In [ ]:
OUTPUT_FILE = "results.json"
with open(OUTPUT_FILE, "w") as f:
    json.dump(results, f, indent=2)

print(f"Saved results to {OUTPUT_FILE}")
print(f"  Autoplay path : {len(results['recommendations']['autoplay_rec'])} videos")
print(f"  Sidebar recs  : {sum(len(s) for s in results['recommendations']['sidebar_rec'])} total links")
print(f"  Restricted    : {len(results['recommendations']['restricted'])} videos")

Saved results to results.json
  Autoplay path : 10 videos
  Sidebar recs  : 87 total links
  Restricted    : 0 videos


## 8. Clean up

In [ ]:
auditor.clean_up()

---

## Appendix: Running a YouTube Shorts audit

Switch `mode` to `"short"` and use Shorts video IDs (the part after `/shorts/` in the URL).
Everything else is identical — the auditor uses `↓` instead of `Shift+N` and collects
`preload_rec` instead of `sidebar_rec`.

In [ ]:
SHORT_SEED_IDS = [
    "VIDEO_ID_1",  # replace with real Shorts IDs
    "VIDEO_ID_2",
]

shorts_auditor = YouTubeAuditor(log_file_path="shorts_demo.log")
shorts_auditor.configure_browser(browser_type="Chrome", headless=True)
shorts_auditor.launch_browser(mode="short", max_duration=10)
shorts_auditor.train(SHORT_SEED_IDS)
shorts_auditor.collect_play_next(collect_video_num=5)

shorts_results = shorts_auditor.report()
print("Shorts audit complete.")
print(json.dumps(shorts_results, indent=2))

shorts_auditor.clean_up()

Shorts audit complete.
{
  "training_ids": ["VIDEO_ID_1"],
  "seed_id": "VIDEO_ID_2",
  "player_mode": "short",
  "max_duration": 10,
  "recommendations": {
    "autoplay_rec": [
      "https://www.youtube.com/shorts/abc123",
      "https://www.youtube.com/shorts/def456"
    ],
    "sidebar_rec": [],
    "preload_rec": [
      ["https://www.youtube.com/shorts/ghi789", "https://www.youtube.com/shorts/jkl012"],
      ["https://www.youtube.com/shorts/mno345", "https://www.youtube.com/shorts/pqr678"]
    ],
    "restricted": []
  }
}
